In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from pathlib import Path
from tqdm.auto import tqdm
from joblib import Parallel, delayed, cpu_count, parallel_backend
from IPython.display import display

from pd_estim_A.data.data_import import (
    load_data, load_ecb_1y_yield,
    fill_liabilities, drop_high_leverage_firms,
    prepare_nig_inputs
)
from pd_estim_A.data.cds_df import get_cds_panel

from pd_estim_A.models.nig.nig_em_afonso import process_one_firm_nig

In [31]:
# Paths
print(Path.cwd())
data_path = Path.cwd() / ".." / "data" / "raw"
output_path = Path.cwd() / ".." / "data" / "derived"

# Load data and prepare panels
ret_daily, bs, coverage = load_data(
    data_path / "Jan2025_Accenture_Dataset_ErasmusCase.xlsx",
    start_date="2012-01-01",
    end_date="2025-12-19",
    enforce_coverage=True,
    coverage_tol=0.995,
    liabilities_scale="auto",
    verbose=True,
)

df_rf = load_ecb_1y_yield(
    startPeriod="2010-01-01",
    endPeriod="2025-12-31",
    out_file= output_path / "ecb_yc_1y_aaa.xml",
    verify_ssl=True,  # recommended if it works
)

df_cal = ret_daily[["date"]].drop_duplicates().sort_values("date").reset_index(drop=True)

debt_daily = fill_liabilities(bs, df_cal)

ret_filt, bs_filt, lev_by_firm, dropped = drop_high_leverage_firms(
    ret_daily,
    bs,
    df_calendar=df_cal,
    debt_daily=debt_daily,
    lev_threshold=8.0,
    lev_agg="median",
    verbose=True,
)

# keep debt panel consistent with filtered firms
keep = set(ret_filt["gvkey"].astype(str).unique())
debt_daily_filt = debt_daily[debt_daily["gvkey"].astype(str).isin(keep)].copy()


nig_df, em_cache = prepare_nig_inputs(ret_filt, bs_filt, df_rf, debt_daily=debt_daily_filt, build_em=False)
print(nig_df.head())
print(nig_df.shape)
print(nig_df.describe())

c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test
[load_data] Firms (ret_daily): 46
[load_data] Date range (ret_daily): 2012-01-03 .. 2025-12-19
[load_data] Coverage min/median/max: 0.999 / 1.000 / 1.000
[load_data] liabilities_scale_used: 1e+06
[load_data] QA mcap_reported<=0 rows (raw windowed mkt): 62
Data has been written to c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\ecb_yc_1y_aaa.xml
[drop_high_leverage_firms] agg=median, threshold=8.0
[drop_high_leverage_firms] firms before: 46 | after: 36
[drop_high_leverage_firms] dropped firms: 10
    gvkey       date             E          isin  \
0  100022 2012-01-03  3.328431e+10  DE0005190003   
1  100080 2012-01-03  4.268705e+10  DE000BAY0017   
2  100312 2012-01-03  1.469717e+09  DE0007030009   
3  100581 2012-01-03  4.935351e+10  FR0000120321   
4  100957 2012-01-03  2.931851e+10  ES0144580Y14   

                        company country_iso         r             L  


In [32]:
# call cds panel and merge
cds = get_cds_panel(
    project_root= Path.cwd() / "..",
    save_csv=False,
    verbose=True,
)
# ensure types
nig = nig_df.copy()
nig["gvkey"] = nig["gvkey"].astype(str)
nig["date"]  = pd.to_datetime(nig["date"])

cds["gvkey"] = cds["gvkey"].astype(str)
cds["date"]  = pd.to_datetime(cds["date"])

# keep only firms that exist in BOTH (drop firms with no CDS)
common_gv = sorted(set(nig["gvkey"].unique()) & set(cds["gvkey"].unique()))
nig = nig[nig["gvkey"].isin(common_gv)].copy()
cds = cds[cds["gvkey"].isin(common_gv)].copy()

# also drop CDS rows whose dates are outside merged's date range
dmin, dmax = nig["date"].min(), nig["date"].max()
cds = cds[(cds["date"] >= dmin) & (cds["date"] <= dmax)].copy()

# merge-asof onto merged's dates (direction='backward')
nig = nig.sort_values(["date", "gvkey"]).reset_index(drop=True)
cds    = cds.sort_values(["date", "gvkey"]).reset_index(drop=True)

merged_cds = pd.merge_asof(
    nig,
    cds,
    on="date",
    by="gvkey",
    direction="backward",
    allow_exact_matches=True,
)

nig_df = merged_cds.reset_index(drop=True)

print("firms after intersection:", nig_df["gvkey"].nunique())
print("rows after merge:", len(nig_df))
print("date range:", nig_df["date"].min(), "→", nig_df["date"].max())

[load_data] Firms (ret_daily): 46
[load_data] Date range (ret_daily): 2012-01-03 .. 2025-12-19
[load_data] Coverage min/median/max: 0.999 / 1.000 / 1.000
[load_data] liabilities_scale_used: 1e+06
[load_data] QA mcap_reported<=0 rows (raw windowed mkt): 62
[get_cds_panel] sheets read: 22 | rows parsed: 67015 | unmapped sheets: 0
firms after intersection: 21
rows after merge: 76508
date range: 2012-01-03 00:00:00 → 2025-12-19 00:00:00


In [ ]:
# Cell 2 — rolling configuration

TRAIN_YEARS = 2
STEP_FREQ = "QE"
WEEK_ENDING = "W-FRI"

T_INV = 1.0                  # 1Y maturity used in inversion
PD_HORIZON_YEARS = 1.0         # 1Y PD horizon on weekly scale

DATA_END = pd.Timestamp("2024-12-31")   # keep equal to Merton if you want clean comparison
LAST_TRAIN_END = DATA_END - pd.offsets.QuarterEnd(1)

MIN_DAILY_ROWS = 10

EM_MIN_ITER = 3
EM_MAX_ITER = 10
EM_TOL = 1e-6

MAX_FIRMS = None
MAX_WINDOWS = None

In [34]:
# Cell 3 — panel preparation
# Assumes nig_df already exists and has at least:
# gvkey, date, E, L, r
# plus optionally company, country_iso, etc.

panel = nig_df.copy()
panel["gvkey"] = panel["gvkey"].astype(str)
panel["date"] = pd.to_datetime(panel["date"])

needed_cols = ["gvkey", "date", "company", "E", "L", "r"]
panel = panel[[c for c in needed_cols if c in panel.columns]].copy()

for c in ["E", "L", "r"]:
    panel[c] = pd.to_numeric(panel[c], errors="coerce")

panel = (
    panel.dropna(subset=["gvkey", "date", "E", "L", "r"])
         .query("E > 0 and L > 0")
         .sort_values(["gvkey", "date"])
         .reset_index(drop=True)
)

firm_daily = {}
for gvkey, g in panel.groupby("gvkey", sort=False):
    g = g.sort_values("date").groupby("date", as_index=False).last()
    firm_daily[gvkey] = g.set_index("date")

gvkeys_all = sorted(firm_daily.keys())
if MAX_FIRMS is not None:
    gvkeys_all = gvkeys_all[:int(MAX_FIRMS)]

print("Firms loaded:", len(firm_daily), "| Firms in run:", len(gvkeys_all))
print("Panel date range:", panel["date"].min().date(), "to", panel["date"].max().date())
print("LAST_TRAIN_END:", LAST_TRAIN_END.date(), "| DATA_END:", DATA_END.date())
display(panel.head())

Firms loaded: 21 | Firms in run: 2
Panel date range: 2012-01-03 to 2025-12-19
LAST_TRAIN_END: 2024-09-30 | DATA_END: 2024-12-31


,gvkey,date,company,E,L,r
0,100022,2012-01-03,BAYERISCHE MOTOREN WERKE AKT,3.328431e+10,8.576700e+10,0.001177
1,100022,2012-01-04,BAYERISCHE MOTOREN WERKE AKT,3.363347e+10,8.576700e+10,0.001037
2,100022,2012-01-05,BAYERISCHE MOTOREN WERKE AKT,3.380203e+10,8.576700e+10,0.001614
3,100022,2012-01-06,BAYERISCHE MOTOREN WERKE AKT,3.344083e+10,8.576700e+10,0.001873
4,100022,2012-01-09,BAYERISCHE MOTOREN WERKE AKT,3.421741e+10,8.576700e+10,0.001835


In [35]:
# Cell 4 — build the rolling window schedule

global_min_date = panel["date"].min()
earliest_end = global_min_date + pd.DateOffset(years=TRAIN_YEARS) - pd.Timedelta(days=1)

train_ends = pd.date_range(start=earliest_end, end=LAST_TRAIN_END, freq=STEP_FREQ)
train_ends = pd.to_datetime(train_ends)

if MAX_WINDOWS is not None:
    train_ends = train_ends[:int(MAX_WINDOWS)]

windows = []
for train_end in train_ends:
    train_start = train_end - pd.DateOffset(years=TRAIN_YEARS) + pd.Timedelta(days=1)
    oos_start = train_end + pd.Timedelta(days=1)
    oos_end = train_end + pd.offsets.QuarterEnd(1)

    windows.append(
        {
            "train_start": pd.Timestamp(train_start),
            "train_end": pd.Timestamp(train_end),
            "oos_start": pd.Timestamp(oos_start),
            "oos_end": pd.Timestamp(oos_end),
        }
    )

windows_df = pd.DataFrame(windows)
display(windows_df.head())
display(windows_df.tail())
print("n_windows:", len(windows_df))

,train_start,train_end,oos_start,oos_end
0,2012-04-01,2014-03-31,2014-04-01,2014-06-30
1,2012-07-01,2014-06-30,2014-07-01,2014-09-30


,train_start,train_end,oos_start,oos_end
0,2012-04-01,2014-03-31,2014-04-01,2014-06-30
1,2012-07-01,2014-06-30,2014-07-01,2014-09-30


n_windows: 2


In [37]:
# Cell 6 — full NIG EM run across firms (parallel across firms with joblib)

def _run_one_firm_nig_joblib(gvkey):
    """
    Worker for one firm.
    Returns:
        {
            "gvkey": ...,
            "result": DataFrame,
            "log": dict
        }
    """
    try:
        df_firm = firm_daily[gvkey].reset_index().copy()

        # loose sanity check on raw daily rows
        if len(df_firm) < MIN_DAILY_ROWS:
            return {
                "gvkey": gvkey,
                "result": pd.DataFrame(),
                "log": {
                    "gvkey": gvkey,
                    "status": "skipped_too_few_daily_rows",
                    "n_rows_daily": len(df_firm),
                    "n_result_rows": 0,
                    "n_ok_rows": 0,
                    "n_fail_rows": 0,
                },
            }

        res_gv = process_one_firm_nig(
            df_firm=df_firm,
            window_plan_df=windows_df,
            train_start_col="train_start",
            train_end_col="train_end",
            oos_start_col="oos_start",
            oos_end_col="oos_end",
            gvkey_col="gvkey",
            input_frequency="daily",   # important
            week_freq=WEEK_ENDING,
            date_col="date",
            equity_col="E",
            debt_col="L",
            rf_col="r",
            start_params=None,         # uses defaults from nig_main
            ann_factor=52.0,
            forecast_horizon_years=T_INV,
            pd_horizon_years=PD_HORIZON_YEARS,
            max_iter=EM_MAX_ITER,
            min_iter=EM_MIN_ITER,
            tol=EM_TOL,
            discounting="continuous",
        )

        if res_gv is None or len(res_gv) == 0:
            n_ok = 0
            n_fail = 0
            n_rows = 0
            status = "empty_output"
            res_out = pd.DataFrame()
        else:
            n_rows = len(res_gv)
            if "ok" in res_gv.columns:
                ok_series = res_gv["ok"].fillna(False).astype(bool)
                n_ok = int(ok_series.sum())
                n_fail = int((~ok_series).sum())
            else:
                n_ok = 0
                n_fail = n_rows
            status = "ok"
            res_out = res_gv

        return {
            "gvkey": gvkey,
            "result": res_out,
            "log": {
                "gvkey": gvkey,
                "status": status,
                "n_rows_daily": len(df_firm),
                "n_result_rows": n_rows,
                "n_ok_rows": n_ok,
                "n_fail_rows": n_fail,
            },
        }

    except Exception as exc:
        return {
            "gvkey": gvkey,
            "result": pd.DataFrame(),
            "log": {
                "gvkey": gvkey,
                "status": f"worker_exception: {exc}",
                "n_rows_daily": np.nan,
                "n_result_rows": 0,
                "n_ok_rows": 0,
                "n_fail_rows": 0,
            },
        }


t0 = time.time()

all_results = []
run_log = []

n_cores = cpu_count()
print(f"Using {n_cores} cores.")

tasks = (delayed(_run_one_firm_nig_joblib)(gvkey) for gvkey in gvkeys_all)

# return_as="generator_unordered" lets results arrive as firms finish,
# so the tqdm bar and the print below update in real time.
with parallel_backend("loky", inner_max_num_threads=1):
    parallel = Parallel(
        n_jobs=-1,
        backend="loky",
        return_as="generator_unordered",
    )

    for i, out in enumerate(
        tqdm(
            parallel(tasks),
            total=len(gvkeys_all),
            desc="Running NIG EM by firm",
            unit="firm",
        ),
        start=1,
    ):
        gvkey = out["gvkey"]
        res_gv = out["result"]
        log_entry = out["log"]

        if res_gv is not None and len(res_gv) > 0:
            all_results.append(res_gv)

        run_log.append(log_entry)

        print(
            f"[{i}/{len(gvkeys_all)}] finished firm {gvkey} "
            f"| ok rows: {log_entry['n_ok_rows']} "
            f"| fail rows: {log_entry['n_fail_rows']}",
            flush=True,
        )

elapsed = time.time() - t0
print(f"\nTotal elapsed time: {elapsed/60:.2f} minutes")

Using 12 cores.


Running NIG EM by firm:   0%|          | 0/2 [00:00<?, ?firm/s]

[1/2] finished firm 100022 | ok rows: 26 | fail rows: 0


Running NIG EM by firm:  50%|█████     | 1/2 [00:11<00:11, 11.94s/firm]

[2/2] finished firm 100080 | ok rows: 26 | fail rows: 0


Running NIG EM by firm: 100%|██████████| 2/2 [00:42<00:00, 21.41s/firm]


Total elapsed time: 0.72 minutes


In [38]:
# Cell 7 — combine outputs and save CSV

if len(all_results) == 0:
    nig_em_afonso = pd.DataFrame()
    print("No results were produced.")
else:
    nig_em_afonso = pd.concat(all_results, ignore_index=True)

    sort_cols = [c for c in ["gvkey", "window_idx", "date"] if c in nig_em_afonso.columns]
    if len(sort_cols) > 0:
        nig_em_afonso = nig_em_afonso.sort_values(sort_cols).reset_index(drop=True)

csv_path = output_path / "nig_em_afonso.csv"
nig_em_afonso.to_csv(csv_path, index=False)

run_log_df = pd.DataFrame(run_log)
display(run_log_df.head())
display(nig_em_afonso.head())

print("Saved results to:", csv_path)
print("Final shape:", nig_em_afonso.shape)
print("Firms in output:", nig_em_afonso["gvkey"].nunique() if "gvkey" in nig_em_afonso.columns else 0)

,gvkey,status,n_rows_daily,n_result_rows,n_ok_rows,n_fail_rows
0,100022,ok,3644,26,26,0
1,100080,ok,3644,26,26,0


,date,A_hat_oos,theta_oos,L_proxy,PD_P,PD_Q,alpha,beta1,delta,beta0,em_converged,em_n_iter,window_train_start,window_train_end,window_oos_start,window_oos_end,gvkey,window_idx,ok,msg
0,2014-04-04,1.590973e+11,-9.792205,1.027250e+11,1.204762e-11,1.251994e-09,271.811197,60.985402,1.425752,-0.274974,True,3,2012-04-06,2014-03-28,2014-04-04,2014-06-27,100022,0,True,ok
1,2014-04-11,1.569189e+11,-9.814829,1.027250e+11,4.316924e-11,3.970244e-09,271.811197,60.985402,1.425752,-0.274974,True,3,2012-04-06,2014-03-28,2014-04-04,2014-06-27,100022,0,True,ok
2,2014-04-18,1.580810e+11,-9.836433,1.027250e+11,2.189901e-11,2.182452e-09,271.811197,60.985402,1.425752,-0.274974,True,3,2012-04-06,2014-03-28,2014-04-04,2014-06-27,100022,0,True,ok
3,2014-04-25,1.565181e+11,-9.798121,1.027250e+11,5.449083e-11,4.854962e-09,271.811197,60.985402,1.425752,-0.274974,True,3,2012-04-06,2014-03-28,2014-04-04,2014-06-27,100022,0,True,ok
4,2014-05-02,1.562258e+11,-9.845144,1.027250e+11,6.455443e-11,5.764901e-09,271.811197,60.985402,1.425752,-0.274974,True,3,2012-04-06,2014-03-28,2014-04-04,2014-06-27,100022,0,True,ok


Saved results to: c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\nig_em_afonso.csv
Final shape: (52, 20)
Firms in output: 2


In [39]:
# Cell 8 — quick diagnostics

if len(nig_em_afonso) > 0:
    if "ok" in nig_em_afonso.columns:
        print(nig_em_afonso["ok"].value_counts(dropna=False))

    cols_show = [c for c in [
        "gvkey", "date", "window_idx", "PD_P", "PD_Q",
        "alpha", "beta1", "delta", "beta0",
        "em_converged", "em_n_iter", "msg"
    ] if c in nig_em_afonso.columns]

    display(nig_em_afonso[cols_show].head(20))

ok
True    52
Name: count, dtype: int64


,gvkey,date,window_idx,PD_P,PD_Q,alpha,beta1,delta,beta0,em_converged,em_n_iter,msg
0,100022,2014-04-04,0,1.204762e-11,1.251994e-09,271.811197,60.985402,1.425752,-0.274974,True,3,ok
1,100022,2014-04-11,0,4.316924e-11,3.970244e-09,271.811197,60.985402,1.425752,-0.274974,True,3,ok
2,100022,2014-04-18,0,2.189901e-11,2.182452e-09,271.811197,60.985402,1.425752,-0.274974,True,3,ok
3,100022,2014-04-25,0,5.449083e-11,4.854962e-09,271.811197,60.985402,1.425752,-0.274974,True,3,ok
4,100022,2014-05-02,0,6.455443e-11,5.764901e-09,271.811197,60.985402,1.425752,-0.274974,True,3,ok
5,100022,2014-05-09,0,8.786944e-11,7.676336e-09,271.811197,60.985402,1.425752,-0.274974,True,3,ok
6,100022,2014-05-16,0,1.560234e-10,1.299416e-08,271.811197,60.985402,1.425752,-0.274974,True,3,ok
7,100022,2014-05-23,0,4.658041e-11,4.475137e-09,271.811197,60.985402,1.425752,-0.274974,True,3,ok
8,100022,2014-05-30,0,2.126645e-11,2.241016e-09,271.811197,60.985402,1.425752,-0.274974,True,3,ok
9,100022,2014-06-06,0,1.597270e-11,1.750516e-09,271.811197,60.985402,1.425752,-0.274974,True,3,ok
